In [1]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv('CO2_Emissions_Canada.csv')
df.sample(5)

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
2494,CHRYSLER,300,FULL-SIZE,3.6,6,A8,X,12.4,7.7,10.3,27,240
2553,FIAT,500 ABARTH HATCHBACK,MINICOMPACT,1.4,4,A6,X,9.6,7.3,8.6,33,200
5459,ACURA,RDX AWD,SUV - SMALL,2.0,4,AS10,Z,11.0,8.6,9.9,29,232
4581,CHEVROLET,EQUINOX AWD,SUV - SMALL,1.6,4,A6,D,8.5,6.1,7.4,38,198
4555,CHEVROLET,COLORADO,PICKUP TRUCK - SMALL,2.5,4,A6,X,12.1,9.2,10.8,26,253


In [5]:
cat_cols = ['Make',	'Model', 'Vehicle Class']

for col in cat_cols:
    df[col] = df[col].str.lower()

df = df.drop_duplicates()

x = df.drop(columns=['CO2 Emissions(g/km)'])
y = df['CO2 Emissions(g/km)']
x_train, x_test, y_train, y_test = train_test_split(x, y , test_size=.2, random_state=42)

In [6]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.6 MB/s eta 0:00:0000:01


In [7]:
!pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 72.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
import dagshub
dagshub.init(
    repo_owner='anni0955', 
    repo_name='CO2-Emission', 
    mlflow=True
)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=4769ded3-6625-42f2-a973-7e07de60bdfc&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c745b960e7f9446a7e8dec93294004c3fc2aa4a2aa8027e2124fe235dc36a828




Accessing as anni0955

Initialized MLflow to track repo "anni0955/CO2-Emission"

Repository anni0955/CO2-Emission initialized!

In [9]:
import mlflow 
mlflow.set_experiment('EXP 2 - without log transforation')

2026/06/15 11:34:51 INFO mlflow.tracking.fluent: Experiment with name 'EXP 2 - without log transforation' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/d193cbbf258d4ad69adac9874e7e5c46', creation_time=1781523291753, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781523291753, lifecycle_stage='active', name='EXP 2 - without log transforation', tags={}, trace_location=None, workspace='default'>

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

import optuna

In [11]:
df.sample(5)

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
3743,gmc,canyon 4wd,pickup truck - small,2.5,4,A6,X,12.7,9.6,11.3,25,263
6732,chevrolet,suburban 4wd,suv - standard,6.2,8,A10,Z,17.2,11.4,14.6,19,343
2529,dodge,dart,mid-size,2.4,4,M6,X,10.5,6.7,8.8,32,205
1315,chevrolet,equinox ffv,suv - small,2.4,4,A6,X,10.5,7.3,9.1,31,209
1725,kia,rondo,station wagon - mid-size,2.0,4,AS6,X,10.6,7.6,9.3,30,214


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler

In [13]:
transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Make', 'Model', 'Vehicle Class', 'Transmission', 'Fuel Type']),
    ('scale', StandardScaler(),['Engine Size(L)', 'Cylinders', 'Fuel Consumption City (L/100 km)', 'Fuel Consumption Hwy (L/100 km)', 'Fuel Consumption Comb (L/100 km)', 'Fuel Consumption Comb (mpg)'])
])

In [14]:
x_train_trans = transformer.fit_transform(x_train)
x_test_trans = transformer.transform(x_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [15]:
x_train_trans.shape

(4792, 1581)

In [16]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [17]:
def objective(trial):
    with mlflow.start_run(nested=True):
        model_name = trial.suggest_categorical('model', ['LR', 'SVR','DT', 'RF', 'GB', 'XGB'])

        if model_name == 'LR':
            model = LinearRegression()

        
        elif model_name == 'SVR':
            kernel_svm = trial.suggest_categorical('kernel_svm', ['linear', 'poly', 'rbf'])

            if kernel_svm == 'linear':
                c_linear = trial.suggest_float('c_linear', 1e-3, 10, log=True)
                model = SVR(C=c_linear, kernel='linear')

            elif kernel_svm == 'poly':
                c_poly = trial.suggest_float('c_poly', 1e-3, 10, log=True)
                degree_poly = trial.suggest_int('degree_poly', 1, 4)
                model = SVR(C=c_poly, degree=degree_poly, kernel='poly')

            else:
                c_rbf = trial.suggest_float('c_rbf', 1e-3, 10, log=True)
                gamma_rbf = trial.suggest_float('gamma_rbf', 1e-3, 10, log=True)
                model = SVR(C=c_rbf, gamma=gamma_rbf, kernel='rbf')

        elif model_name == 'DT':
            criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'absolute_error', 'poisson', 'squared_error'])
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = DecisionTreeRegressor(criterion=criterion, max_depth=max_depth)

        elif model_name == 'RF':
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
        
        elif model_name == 'GB':
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            learning_rate = trial.suggest_float('learning_rate', 1e-2, 1)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)

        else:
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            learning_rate = trial.suggest_float('learning_rate', 1e-2, 1)
            max_depth = trial.suggest_int('max_depth', 4, 20)
            model = XGBRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42, n_jobs=-1)    

        
        model.fit(x_train_trans, y_train)
        mlflow.log_params(trial.params)

        test_y_pred = model.predict(x_test_trans)
        train_y_pred = model.predict(x_train_trans)

        test_MAE = mean_absolute_error(y_test, test_y_pred)
        train_MAE = mean_absolute_error(y_train, train_y_pred)

        test_RMSE = root_mean_squared_error(y_test, test_y_pred)
        train_RMSE = root_mean_squared_error(y_train, train_y_pred)

        mlflow.log_metric('train_MAE', train_MAE)
        mlflow.log_metric('test_MAE', test_MAE)

        mlflow.log_metric('train_RMSE', train_RMSE)
        mlflow.log_metric('test_RMSE', test_RMSE)

        return test_RMSE


In [18]:
study = optuna.create_study(direction='minimize', study_name='Model Selection')

with mlflow.start_run(run_name='Best Model') as parent:
    study.optimize(objective, n_trials=50, n_jobs=-1)
    mlflow.log_params(study.best_params)

    mlflow.log_metric('best_score', study.best_value)

[I 2026-06-15 11:35:47,244] A new study created in memory with name: Model Selection
[I 2026-06-15 11:35:59,440] Trial 0 finished with value: 21.909942381005294 and parameters: {'model': 'SVR', 'kernel_svm': 'linear', 'c_linear': 0.003710111866700193}. Best is trial 0 with value: 21.909942381005294.


🏃 View run zealous-mare-999 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1/runs/3dc59e2dbcbe47b98d61bbf21f8c33ab
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1


[I 2026-06-15 11:36:01,211] Trial 1 finished with value: 4.821113109588623 and parameters: {'model': 'XGB', 'n_estimators': 345, 'learning_rate': 0.8915000956031599, 'max_depth': 13}. Best is trial 1 with value: 4.821113109588623.


🏃 View run clean-boar-342 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1/runs/263853bea3d6409bb4036d41f9c61399
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1


[I 2026-06-15 11:36:25,945] Trial 3 finished with value: 4.414655936103098 and parameters: {'model': 'RF', 'n_estimators': 233, 'max_depth': 8}. Best is trial 3 with value: 4.414655936103098.


...

🏃 View run mysterious-slug-317 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1/runs/82d32d9a3f2848d9a9af62cc27d1eab8
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1


[I 2026-06-15 12:05:23,594] Trial 46 finished with value: 4.205364978844793 and parameters: {'model': 'RF', 'n_estimators': 361, 'max_depth': 17}. Best is trial 45 with value: 4.2043193857439425.
[I 2026-06-15 12:07:32,654] Trial 48 finished with value: 4.2697988538845 and parameters: {'model': 'RF', 'n_estimators': 343, 'max_depth': 16}. Best is trial 45 with value: 4.2043193857439425.


🏃 View run dapper-fox-943 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1/runs/71998c4cc90c40bf86ac23ec41a05732
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1


[I 2026-06-15 12:07:51,903] Trial 49 finished with value: 4.268716894052171 and parameters: {'model': 'RF', 'n_estimators': 335, 'max_depth': 16}. Best is trial 45 with value: 4.2043193857439425.


🏃 View run funny-conch-734 at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1/runs/cabb6e5f699c4d378caade0f7064fbd1
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1
🏃 View run Best Model at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1/runs/3a9b3885f7eb40c98833159b492c8b7f
🧪 View experiment at: https://dagshub.com/anni0955/CO2-Emission.mlflow/#/experiments/1
